In [1]:
!pip show chromadb rank_bm25 sentence-transformers | grep -E "Name|Version"

Name: sentence-transformers
Version: 5.4.0


In [2]:
!pip install -q chromadb rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 76.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 111.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [3]:
import os
import shutil

src = '/kaggle/input/datasets/harshamin07/researchlens-500'
dst = '/kaggle/working/'

os.makedirs(dst, exist_ok=True)

for filename in os.listdir(src):
    full_src = os.path.join(src, filename)
    if os.path.isfile(full_src):
        shutil.copy(full_src, os.path.join(dst, filename))

print("Loaded from dataset:")
for f in os.listdir(dst):
    size_mb = os.path.getsize(os.path.join(dst, f)) / (1024 * 1024)
    print(f"  ✓ {f} — {size_mb:.2f} MB")

Loaded from dataset:
  ✓ papers_metadata.json — 0.77 MB
  ✓ qasper_pairs.json — 0.06 MB
  ✓ parsed_papers.json — 33.57 MB
  ✓ .virtual_documents — 0.00 MB


In [4]:
import json
import re
import pickle
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np

In [8]:
with open('/kaggle/working/parsed_papers.json', 'r') as f:
    all_pages = json.load(f)

print(f"Total pages loaded: {len(all_pages)}")
print(f"Sample keys: {all_pages[0].keys()}")
print(f"Sample text (first 200 chars): {all_pages[0]['text'][:200]}")

Total pages loaded: 8565
Sample keys: dict_keys(['paper_id', 'title', 'page_num', 'text'])
Sample text (first 200 chars): ENHANCING HUMAN-LIKE RESPONSES IN LARGE LANGUAGE
MODELS∗
Ethem Ya˘gız Çalık
X: @Weyaxi
Hugging Face: huggingface.co/Weyaxi
ethemyagiz1@gmail.com
Talha Rüzgar Akku¸s
X: @qbert_ai
Hugging Face: huggingf


In [9]:
# ── Action 2.2 — Reference-Section Filtering ─────────────────────────────────
# Applied BEFORE clean_text() so that [\d+] bracket citations are still present
# and detectable. Two-pass: page-level cascade first, chunk-level density second.
from collections import defaultdict

# Tune these if the filter proves too aggressive or too lenient
DENSITY_THRESHOLD     = 0.40   # >40% of segments look like citations → drop chunk
MIN_LINES_FOR_DENSITY = 4      # don't density-check very short chunks

# Patterns that flag a line/segment as a bibliography entry
_CITATION_LINE_PATTERNS = [
    re.compile(r'\[\d+\]'),                            # [1], [23]
    re.compile(r'\bet\s+al\.', re.I),                  # et al.
    re.compile(r'arXiv\s*:\s*\d{4}\.\d{4,5}', re.I),  # arXiv:2201.12345
    re.compile(r'\bdoi\s*:', re.I),                    # doi:
    re.compile(r'pp\.\s*\d+[-–]\d+'),                  # pp. 12-34
    re.compile(r'(?:vol\.|volume|issue|no\.)\s*\d+', re.I),  # vol. 3, no. 2
]

# A standalone header line that signals a reference section starts here
_REF_HEADER_RE = re.compile(
    r'^\s*(?:references?|bibliography|works\s+cited)\s*$',
    re.I | re.M
)

def _is_citation_line(line):
    line = line.strip()
    if len(line) < 5:
        return False
    return any(pat.search(line) for pat in _CITATION_LINE_PATTERNS)

def _page_starts_references(text):
    """True if the raw page text contains a standalone reference-section header."""
    return bool(_REF_HEADER_RE.search(text))

def _split_at_references(text):
    """Return the content BEFORE the first reference-section header.

    Returns None if that content is too short to be worth keeping (< 80 chars),
    e.g. when the header is the very first line of the page.
    """
    match = _REF_HEADER_RE.search(text)
    if match is None:
        return None
    before = text[:match.start()].strip()
    return before if len(before) >= 80 else None

def chunk_citation_density(text):
    """Fraction of segments in a chunk that look like citation entries.

    Tries newline-split first (raw text). Falls back to sentence-split on '. '
    for cleaned single-line text where newlines were collapsed by clean_text().
    """
    lines = [l for l in text.splitlines() if l.strip()]
    if len(lines) < MIN_LINES_FOR_DENSITY:
        lines = [s.strip() for s in re.split(r'\.\s+', text) if s.strip()]
    if len(lines) < MIN_LINES_FOR_DENSITY:
        return 0.0
    return sum(1 for l in lines if _is_citation_line(l)) / len(lines)


# ── Page-level cascade filter ─────────────────────────────────────────────────
# For each paper: find the first page whose text contains a reference-section
# header. Keep content BEFORE the header on that page (conclusion/code-avail
# sections often share a page with the references), then drop that page and
# all subsequent pages via cascade.

pages_by_paper = defaultdict(list)
for page in all_pages:
    pages_by_paper[page['paper_id']].append(page)
for pid in pages_by_paper:
    pages_by_paper[pid].sort(key=lambda p: p['page_num'])

pages_before    = len(all_pages)
kept_pages      = []
dropped_pages   = []
salvaged_count  = 0   # cutoff pages where pre-header content was kept

for pid, pages in pages_by_paper.items():
    cutoff = None
    for i, page in enumerate(pages):
        if _page_starts_references(page['text']):
            cutoff = i
            break

    if cutoff is not None:
        # Keep all pages before the cutoff unchanged
        kept_pages.extend(pages[:cutoff])

        # Salvage content from the cutoff page that appears BEFORE the header
        # (catches: conclusion + references on same page, code availability, etc.)
        before_text = _split_at_references(pages[cutoff]['text'])
        if before_text:
            kept_pages.append({**pages[cutoff], 'text': before_text})
            salvaged_count += 1

        # Track dropped pages for sanity-check display
        dropped_pages.extend(pages[cutoff:])
    else:
        kept_pages.extend(pages)

net_dropped = pages_before - len(kept_pages)
print(f"=== Page-Level Reference Filter ===")
print(f"  Pages before       : {pages_before}")
print(f"  Cutoff pages found : {len(dropped_pages) - (sum(1 for p in dropped_pages if not _page_starts_references(p['text'])))}")
print(f"  Cutoff pages salvaged (pre-header content kept): {salvaged_count}")
print(f"  Net pages dropped  : {net_dropped}  ({net_dropped / pages_before * 100:.1f}%)")
print(f"  Pages kept         : {len(kept_pages)}")

print(f"\n--- Sample dropped pages (first 5 reference-header pages) ---")
shown = 0
for page in dropped_pages:
    if _page_starts_references(page['text']) and shown < 5:
        preview = page['text'][:350].replace('\n', ' ')
        salvaged = _split_at_references(page['text'])
        print(f"\n  Paper    : {page['title'][:65]}")
        print(f"  PageNo   : {page['page_num']}")
        print(f"  Salvaged : {len(salvaged)} chars before header" if salvaged else "  Salvaged : (none — header at top of page)")
        print(f"  Full text: {preview} ...")
        shown += 1

# Pass filtered list to the next cell (clean_text → chunking)
all_pages = kept_pages
print(f"\nall_pages updated → {len(all_pages)} pages")

=== Page-Level Reference Filter ===
  Pages before       : 8565
  Cutoff pages found : 510
  Cutoff pages salvaged (pre-header content kept): 388
  Net pages dropped  : 3340  (39.0%)
  Pages kept         : 5225

--- Sample dropped pages (first 5 reference-header pages) ---

  Paper    : Enhancing Human-Like Responses in Large Language Models
  PageNo   : 8
  Salvaged : 2001 chars before header
  Full text: Enhancing Human-Like Responses in Large Language Models We have published our models and dataset on Hugging Face [9] to support further research and development in this field. The models can be accessed through the following links: • HumanLLMs/Human-Like-LLama3-8B-Instruct • HumanLLMs/Human-Like-Qwen2.5-7B-Instruct • HumanLLMs/Human-Like-Mistral-Ne ...

  Paper    : Is Self-knowledge and Action Consistent or Not: Investigating Lar
  PageNo   : 4
  Salvaged : 2011 chars before header
  Full text: Is Self-knowledge and Action Consistent or Not: Investigating Large Language Model’s Pers

In [10]:
def clean_text(text):
    text = re.sub(r'-\n', '', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'BIBREF\d+', '', text)
    text = re.sub(r'FIGREF\d+', '', text)
    text = re.sub(r'TABREF\d+', '', text)
    return text.strip()

all_pages_clean = []
for page in all_pages:
    cleaned = clean_text(page['text'])
    if len(cleaned) > 100:
        all_pages_clean.append({
            'paper_id': page['paper_id'],
            'title': page['title'],
            'page_num': page['page_num'],
            'text': cleaned
        })

print(f"Pages after cleaning: {len(all_pages_clean)}")
print(f"Sample cleaned text: {all_pages_clean[0]['text'][:200]}")

Pages after cleaning: 5217
Sample cleaned text: ENHANCING HUMAN-LIKE RESPONSES IN LARGE LANGUAGE MODELS∗ Ethem Ya˘gız Çalık X: @Weyaxi Hugging Face: huggingface.co/Weyaxi ethemyagiz1@gmail.com Talha Rüzgar Akku¸s X: @qbert_ai Hugging Face: huggingf


In [15]:
def fixed_chunk(text, paper_id, title, page_num, chunk_size=512, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    chunk_idx = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk_text = ' '.join(words[start:end])
        
        if len(chunk_text) > 100:
            chunks.append({
                'chunk_id': f"{paper_id}_p{page_num}_c{chunk_idx}",
                'paper_id': paper_id,
                'title': title,
                'page_num': page_num,
                'text': chunk_text,
                'chunk_type': 'fixed'
            })
            chunk_idx += 1
        
        start += chunk_size - overlap
    
    return chunks

fixed_chunks = []
for page in all_pages_clean:
    chunks = fixed_chunk(
        page['text'],
        page['paper_id'],
        page['title'],
        page['page_num']
    )
    fixed_chunks.extend(chunks)

print(f"Total fixed chunks: {len(fixed_chunks)}")
print(f"Average chunk length (words): {np.mean([len(c['text'].split()) for c in fixed_chunks]):.0f}")

Total fixed chunks: 8933
Average chunk length (words): 349


In [16]:
def semantic_chunk(text, paper_id, title, page_num, max_words=400, min_words=50):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current_sentences = []
    current_word_count = 0
    chunk_idx = 0
    
    for sentence in sentences:
        word_count = len(sentence.split())
        
        if current_word_count + word_count > max_words and current_word_count >= min_words:
            chunk_text = ' '.join(current_sentences)
            chunks.append({
                'chunk_id': f"{paper_id}_p{page_num}_s{chunk_idx}",
                'paper_id': paper_id,
                'title': title,
                'page_num': page_num,
                'text': chunk_text,
                'chunk_type': 'semantic'
            })
            chunk_idx += 1
            current_sentences = [sentence]
            current_word_count = word_count
        else:
            current_sentences.append(sentence)
            current_word_count += word_count
    
    if current_sentences and current_word_count >= min_words:
        chunks.append({
            'chunk_id': f"{paper_id}_p{page_num}_s{chunk_idx}",
            'paper_id': paper_id,
            'title': title,
            'page_num': page_num,
            'text': ' '.join(current_sentences),
            'chunk_type': 'semantic'
        })
    
    return chunks

semantic_chunks = []
for page in all_pages_clean:
    chunks = semantic_chunk(
        page['text'],
        page['paper_id'],
        page['title'],
        page['page_num']
    )
    semantic_chunks.extend(chunks)

print(f"Total semantic chunks: {len(semantic_chunks)}")
print(f"Average chunk length (words): {np.mean([len(c['text'].split()) for c in semantic_chunks]):.0f}")

Total semantic chunks: 9561
Average chunk length (words): 306


In [17]:
# ── Chunk-Level Density Second Pass ──────────────────────────────────────────
# DISABLED: The page-level cascade (cell-5) removes all bibliography pages.
# A density pass using `et al.` as a signal fires on body-text paragraphs in
# citation-heavy papers (surveys, RAG papers) — false positive rate too high.
# Re-enable if page-level filtering proves insufficient after evaluation.

print(f"=== Chunk-Level Density Filter ===")
print(f"  Disabled — page-level cascade handles reference removal.")
print(f"  Fixed   : {len(fixed_chunks)}")
print(f"  Semantic: {len(semantic_chunks)}")

=== Chunk-Level Density Filter ===
  Disabled — page-level cascade handles reference removal.
  Fixed   : 8933
  Semantic: 9561


In [18]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

test_embedding = embed_model.encode("test sentence")
print(f"Embedding model loaded")
print(f"Embedding dimension: {len(test_embedding)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded
Embedding dimension: 384


In [19]:
all_chunks = fixed_chunks + semantic_chunks
all_texts = [c['text'] for c in all_chunks]

print(f"Embedding {len(all_texts)} chunks total...")

embeddings = embed_model.encode(
    all_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

fixed_embeddings = embeddings[:len(fixed_chunks)]
semantic_embeddings = embeddings[len(fixed_chunks):]

print(f"Fixed chunk embeddings shape: {fixed_embeddings.shape}")
print(f"Semantic chunk embeddings shape: {semantic_embeddings.shape}")

Embedding 18494 chunks total...


Batches:   0%|          | 0/578 [00:00<?, ?it/s]

Fixed chunk embeddings shape: (8933, 384)
Semantic chunk embeddings shape: (9561, 384)


In [20]:
chroma_client = chromadb.PersistentClient(path='/kaggle/working/chroma_db')

try:
    chroma_client.delete_collection("fixed_chunks")
    chroma_client.delete_collection("semantic_chunks")
except:
    pass

fixed_collection = chroma_client.create_collection(
    name="fixed_chunks",
    metadata={"hnsw:space": "cosine"}
)

semantic_collection = chroma_client.create_collection(
    name="semantic_chunks",
    metadata={"hnsw:space": "cosine"}
)

print("Collections created")

Collections created


In [21]:
batch_size = 500

for i in range(0, len(fixed_chunks), batch_size):
    batch_chunks = fixed_chunks[i:i+batch_size]
    batch_embeddings = fixed_embeddings[i:i+batch_size]
    
    fixed_collection.add(
        ids=[c['chunk_id'] for c in batch_chunks],
        documents=[c['text'] for c in batch_chunks],
        embeddings=batch_embeddings.tolist(),
        metadatas=[{
            'paper_id': c['paper_id'],
            'title': c['title'],
            'page_num': c['page_num'],
            'chunk_type': c['chunk_type']
        } for c in batch_chunks]
    )
    print(f"Fixed: added batch {i//batch_size + 1}/{(len(fixed_chunks)-1)//batch_size + 1}")

print(f"\nFixed collection total: {fixed_collection.count()}")

Fixed: added batch 1/18
Fixed: added batch 2/18
Fixed: added batch 3/18
Fixed: added batch 4/18
Fixed: added batch 5/18
Fixed: added batch 6/18
Fixed: added batch 7/18
Fixed: added batch 8/18
Fixed: added batch 9/18
Fixed: added batch 10/18
Fixed: added batch 11/18
Fixed: added batch 12/18
Fixed: added batch 13/18
Fixed: added batch 14/18
Fixed: added batch 15/18
Fixed: added batch 16/18
Fixed: added batch 17/18
Fixed: added batch 18/18

Fixed collection total: 8933


In [23]:
for i in range(0, len(semantic_chunks), batch_size):
    batch_chunks = semantic_chunks[i:i+batch_size]
    batch_embeddings = semantic_embeddings[i:i+batch_size]
    
    semantic_collection.add(
        ids=[c['chunk_id'] for c in batch_chunks],
        documents=[c['text'] for c in batch_chunks],
        embeddings=batch_embeddings.tolist(),
        metadatas=[{
            'paper_id': c['paper_id'],
            'title': c['title'],
            'page_num': c['page_num'],
            'chunk_type': c['chunk_type']
        } for c in batch_chunks]
    )
    print(f"Semantic: added batch {i//batch_size + 1}/{(len(semantic_chunks)-1)//batch_size + 1}")

print(f"\nSemantic collection total: {semantic_collection.count()}")

Semantic: added batch 1/20
Semantic: added batch 2/20
Semantic: added batch 3/20
Semantic: added batch 4/20
Semantic: added batch 5/20
Semantic: added batch 6/20
Semantic: added batch 7/20
Semantic: added batch 8/20
Semantic: added batch 9/20
Semantic: added batch 10/20
Semantic: added batch 11/20
Semantic: added batch 12/20
Semantic: added batch 13/20
Semantic: added batch 14/20
Semantic: added batch 15/20
Semantic: added batch 16/20
Semantic: added batch 17/20
Semantic: added batch 18/20
Semantic: added batch 19/20
Semantic: added batch 20/20

Semantic collection total: 9561


In [24]:
bm25_corpus = [c['text'].lower().split() for c in fixed_chunks]
bm25_index = BM25Okapi(bm25_corpus)

bm25_metadata = [{
    'chunk_id': c['chunk_id'],
    'paper_id': c['paper_id'],
    'title': c['title'],
    'page_num': c['page_num'],
    'text': c['text']
} for c in fixed_chunks]

print(f"BM25 index built on {len(bm25_corpus)} documents")
print(f"Vocabulary size: {len(bm25_index.idf)}")

BM25 index built on 8933 documents
Vocabulary size: 184347


In [25]:
test_query = "attention mechanism in transformers"
test_embedding = embed_model.encode(test_query).tolist()

chroma_results = fixed_collection.query(
    query_embeddings=[test_embedding],
    n_results=3,
    include=['documents', 'distances', 'metadatas']
)

print("ChromaDB top 3 results:")
for i, (doc, dist, meta) in enumerate(zip(
    chroma_results['documents'][0],
    chroma_results['distances'][0],
    chroma_results['metadatas'][0]
)):
    print(f"\n  [{i+1}] Score: {1-dist:.3f} | Paper: {meta['title'][:60]}")
    print(f"       Text: {doc[:150]}")

bm25_scores = bm25_index.get_scores(test_query.lower().split())
top_bm25_idx = np.argsort(bm25_scores)[::-1][:3]

print("\n\nBM25 top 3 results:")
for i, idx in enumerate(top_bm25_idx):
    print(f"\n  [{i+1}] Score: {bm25_scores[idx]:.3f} | Paper: {bm25_metadata[idx]['title'][:60]}")
    print(f"       Text: {bm25_metadata[idx]['text'][:150]}")

ChromaDB top 3 results:

  [1] Score: 0.689 | Paper: Self-Attention as Distributional Projection: A Unified Inter
       Text: a broader, trainable framework. 8 Discussion and Implications This paper has presented a unified mathematical interpretation of self-attention through

  [2] Score: 0.666 | Paper: A Self-Supervised Reinforcement Learning Approach for Fine-T
       Text: patterns, and minimal repetition, all without requiring external human feedback. This approach leverages the inherent interpretability of cross-attent

  [3] Score: 0.632 | Paper: Self-Attention as Distributional Projection: A Unified Inter
       Text: Self-Attention as Distributional Projection: A Unified Interpretation of Transformer Architecture Satt = QattK⊤ att = HWQW ⊤ KH⊤ generalize our earlie


BM25 top 3 results:

  [1] Score: 16.475 | Paper: Attention as a Hypernetwork
       Text: Published as a conference paper at ICLR 2025 Figure 1: Hypernetwork attention. A A linear hypernetwork maps a latent code

In [26]:
with open('/kaggle/working/bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25_index, f)

with open('/kaggle/working/bm25_metadata.pkl', 'wb') as f:
    pickle.dump(bm25_metadata, f)

with open('/kaggle/working/fixed_chunks.pkl', 'wb') as f:
    pickle.dump(fixed_chunks, f)

with open('/kaggle/working/semantic_chunks.pkl', 'wb') as f:
    pickle.dump(semantic_chunks, f)

print("Saved:")
files = ['bm25_index.pkl', 'bm25_metadata.pkl', 'fixed_chunks.pkl', 'semantic_chunks.pkl']
for fname in files:
    size_mb = os.path.getsize(f'/kaggle/working/{fname}') / (1024*1024)
    print(f"  ✓ {fname} — {size_mb:.2f} MB")

chroma_db_size = sum(
    os.path.getsize(os.path.join(root, file))
    for root, dirs, files_list in os.walk('/kaggle/working/chroma_db')
    for file in files_list
) / (1024*1024)
print(f"  ✓ chroma_db/ — {chroma_db_size:.2f} MB")

Saved:
  ✓ bm25_index.pkl — 22.89 MB
  ✓ bm25_metadata.pkl — 20.85 MB
  ✓ fixed_chunks.pkl — 20.88 MB
  ✓ semantic_chunks.pkl — 19.67 MB
  ✓ chroma_db/ — 336.05 MB


In [27]:
print("=== Notebook 02 Complete ===\n")
print(f"Fixed chunks:    {len(fixed_chunks)}")
print(f"Semantic chunks: {len(semantic_chunks)}")
print(f"ChromaDB fixed:  {fixed_collection.count()}")
print(f"ChromaDB sem:    {semantic_collection.count()}")
print(f"BM25 docs:       {len(bm25_corpus)}")

=== Notebook 02 Complete ===

Fixed chunks:    8933
Semantic chunks: 9561
ChromaDB fixed:  8933
ChromaDB sem:    9561
BM25 docs:       8933
